In [3]:
import graphviz
import os

# Helper function to round long floats in the feature names for readability
def fmt_num(num_str):
    try:
        return f"{float(num_str):.4f}"
    except ValueError:
        return num_str

def simulate_decision_tree_graph():
    # Initialize Digraph
    dot = graphviz.Digraph('DecisionTree', comment='Manually Reconstructed Tree')
    
    # Global graph attributes for cleaner look with higher resolution
    dot.attr(rankdir='TB', size='20,20', dpi='300')
    dot.attr('node', shape='box', style='filled', color='lightblue2', fontname='Helvetica', fontsize='12')
    dot.attr('edge', fontname='Helvetica', fontsize='12')

    # Helper to create leaf nodes differently
    leaf_counter = 0
    def add_leaf(prob):
        nonlocal leaf_counter
        node_id = f'leaf_{leaf_counter}'
        # Color leaves based on probability (lighter for low, darker for high)
        gray_level = int(255 - (prob * 200))
        hex_color = f'#{gray_level:02x}{gray_level:02x}{gray_level:02x}'
        
        label = f'Probability:\n{prob:.4f}'
        dot.node(node_id, label, shape='box', style='rounded,filled', 
                 color='black', fillcolor=hex_color, fontcolor='white' if prob > 0.5 else 'black')
        leaf_counter += 1
        return node_id

    # ---- Tree Construction Starts Here ----

    # Level 0 Root
    node_root = 'n0'
    dot.node(node_root, f'sf_low_episode_count <= 47.5')

    # Level 1 (Left Branch - True)
    n1_L = 'n1_L'
    dot.node(n1_L, f'peak_fentanyl_rate_per_sec <=\n{fmt_num(0.05495169013738632)}')
    dot.edge(node_root, n1_L, label='True')

    # Level 2 (Left-Left)
    n2_LL = 'n2_LL'
    dot.node(n2_LL, f'shock_index_avg <=\n{fmt_num(0.7405163049697876)}')
    dot.edge(n1_L, n2_LL, label='True')

    # Level 3 (Left-Left-Left)
    n3_LLL = 'n3_LLL'
    dot.node(n3_LLL, f'sf_low_episode_count <= 8.5')
    dot.edge(n2_LL, n3_LLL, label='True')

    # Leaf LLLL
    leaf_llll = add_leaf(0.0)
    dot.edge(n3_LLL, leaf_llll, label='True')
    
    # Node LLLR (Else)
    n4_LLLR = 'n4_LLLR'
    dot.node(n4_LLLR, f'shock_index_avg <=\n{fmt_num(0.6816148161888123)}')
    dot.edge(n3_LLL, n4_LLLR, label='False')
    # Leaves
    dot.edge(n4_LLLR, add_leaf(0.0), label='True')
    dot.edge(n4_LLLR, add_leaf(0.8808483290488431), label='False')

    # Level 3 (Left-Left-Right - Else)
    n3_LLR = 'n3_LLR'
    dot.node(n3_LLR, f'sf_low_total_duration_sec <=\n47970.0')
    dot.edge(n2_LL, n3_LLR, label='False')

    # Level 4 (LLRL)
    n4_LLRL = 'n4_LLRL'
    dot.node(n4_LLRL, f'sf_cv <= {fmt_num(0.20176851749420166)}')
    dot.edge(n3_LLR, n4_LLRL, label='True')
    # Leaves
    dot.edge(n4_LLRL, add_leaf(0.03691135995174004), label='True')
    dot.edge(n4_LLRL, add_leaf(0.2856533644566137), label='False')

    # Level 4 (LLRR - Else)
    n4_LLRR = 'n4_LLRR'
    dot.node(n4_LLRR, f'sf_cv <= {fmt_num(0.17625226825475693)}')
    dot.edge(n3_LLR, n4_LLRR, label='False')
    # Leaves
    dot.edge(n4_LLRR, add_leaf(0.7917509098261221), label='True')
    dot.edge(n4_LLRR, add_leaf(0.0), label='False')

    # Level 2 (Left-Right - Else)
    n2_LR = 'n2_LR'
    dot.node(n2_LR, f'med_load_fentanyl_per_hr <=\n{fmt_num(97.54008865356445)}')
    dot.edge(n1_L, n2_LR, label='False')

    # Level 3 (LRL)
    n3_LRL = 'n3_LRL'
    dot.node(n3_LRL, f'sf_low_episode_count <= 38.5')
    dot.edge(n2_LR, n3_LRL, label='True')

    # Level 4 (LRLL)
    n4_LRLL = 'n4_LRLL'
    dot.node(n4_LRLL, f'med_load_fentanyl_per_hr <=\n{fmt_num(34.700504302978516)}')
    dot.edge(n3_LRL, n4_LRLL, label='True')
    # Leaves
    dot.edge(n4_LRLL, add_leaf(0.6726788628360393), label='True')
    dot.edge(n4_LRLL, add_leaf(0.4445705852301759), label='False')

    # Leaf LRLR (Else)
    dot.edge(n3_LRL, add_leaf(0.0), label='False')

    # Leaf LRR (Else)
    dot.edge(n2_LR, add_leaf(0.0), label='False')


    # Level 1 (Right Branch - False / Else from Root)
    n1_R = 'n1_R'
    dot.node(n1_R, f'med_load_fentanyl_per_hr <=\n{fmt_num(89.03385925292969)}')
    dot.edge(node_root, n1_R, label='False')

    # Level 2 (RL)
    n2_RL = 'n2_RL'
    dot.node(n2_RL, f'high_fio2_episode_count <= 32.0')
    dot.edge(n1_R, n2_RL, label='True')

    # Level 3 (RLL)
    n3_RLL = 'n3_RLL'
    dot.node(n3_RLL, f'map_std <= {fmt_num(10.754669666290283)}')
    dot.edge(n2_RL, n3_RLL, label='True')

    # Level 4 (RLLL)
    n4_RLLL = 'n4_RLLL'
    dot.node(n4_RLLL, f'sf_cv <= {fmt_num(0.1733771413564682)}')
    dot.edge(n3_RLL, n4_RLLL, label='True')
    # Leaves
    dot.edge(n4_RLLL, add_leaf(0.7600931677018633), label='True')
    dot.edge(n4_RLLL, add_leaf(0.0), label='False')

    # Level 4 (RLLR - Else)
    n4_RLLR = 'n4_RLLR'
    dot.node(n4_RLLR, f'peak_fentanyl_rate_per_sec <=\n{fmt_num(0.0486111119389534)}')
    dot.edge(n3_RLL, n4_RLLR, label='False')
    # Leaves
    dot.edge(n4_RLLR, add_leaf(0.8342170043552357), label='True')
    dot.edge(n4_RLLR, add_leaf(0.9394230173961493), label='False')

    # Level 3 (RLR - Else)
    n3_RLR = 'n3_RLR'
    dot.node(n3_RLR, f'sf_cv <= {fmt_num(0.30367089807987213)}')
    dot.edge(n2_RL, n3_RLR, label='False')
    # Leaves
    dot.edge(n3_RLR, add_leaf(0.7600931677018633), label='True')
    dot.edge(n3_RLR, add_leaf(0.0), label='False')
    
    # Leaf RR (Else)
    dot.edge(n1_R, add_leaf(0.0), label='False')

    # Render the graph
    # Save both .gv and .png files without trying to view automatically
    try:
        # First create the output directory if it doesn't exist
        output_dir = '/workspaces/BrainFlux'
        os.makedirs(output_dir, exist_ok=True)
        
        # Set the output path
        output_path = os.path.join(output_dir, 'decision_tree')
        
        # Render without automatic viewing (view=False for container environments)
        dot.render(output_path, view=False, format='png', cleanup=True)
        
        # Also save as SVG for better scalability
        dot.render(output_path, view=False, format='svg', cleanup=True)
        
        print(f"Decision tree visualization generated successfully!")
        print(f"PNG file: {output_path}.png")
        print(f"SVG file: {output_path}.svg")
        
        # Display the source code for debugging if needed
        print(f"\nGraphviz source saved to: {output_path}")
        
    except Exception as e:
        print(f"Error generating decision tree visualization: {e}")
        
        # Try to save just the .gv file if rendering fails
        try:
            output_path = os.path.join(output_dir, 'decision_tree')
            with open(f'{output_path}.gv', 'w') as f:
                f.write(dot.source)
            print(f"Graphviz source code saved to: {output_path}.gv")
            print("You can manually render this file using: dot -Tpng decision_tree.gv -o decision_tree.png")
        except Exception as save_error:
            print(f"Error saving Graphviz source: {save_error}")
            print("Printing Graphviz source code:")
            print(dot.source)

# Run the function
simulate_decision_tree_graph()

Decision tree visualization generated successfully!
PNG file: /workspaces/BrainFlux/decision_tree.png
SVG file: /workspaces/BrainFlux/decision_tree.svg

Graphviz source saved to: /workspaces/BrainFlux/decision_tree
